# LangGraph G11 — MCP: tools from servers
So far every tool was a Python function in this notebook. In a real organisation the library
system, the payments system and the calendar are owned by other teams and run as separate
services. **MCP (Model Context Protocol)** is the open standard for exposing tools (and
resources and prompts) from a server so that *any* agent can use them:

```text
CampusAI graph  --MCP client-->  library server (separate process, HTTP)  --> its own code and data
   (tools node)                   exposes: library_hours, search_library
```

An MCP server lists its tools with names, descriptions and schemas, exactly the shape the model
already reads. The client turns them into tool objects that go straight into `ToolNode`. Two
consequences: the ecosystem of tools is decoupled from your agent, and **a server is code you
run on the model's behalf, so trust it as you would trust a library**: read what it exposes,
and keep the guard and approval layers of G8 in front of its write tools.

MCP servers speak over **stdio** (started as a subprocess) or **HTTP** (running as a service).
We use HTTP, the shape of a real deployment: the server starts once in the background and any
number of agents connect to its URL. MCP tools are asynchronous, so this section runs the graph
with `ainvoke` (a notebook cell may use `await` directly).

In [ ]:
%pip install -q -U mcp langchain-mcp-adapters

### Step 1 — Write a small MCP server (it would normally be another team's code)

`FastMCP` turns plain functions into MCP tools. The file is written to disk and started as a
background process serving HTTP on a local port.

In [ ]:
%%writefile campus_library_server.py
# ours: a minimal MCP server. In production this would be a service owned by the library team.
from mcp.server.fastmcp import FastMCP          # mcp: the reference Python SDK

server = FastMCP("campus-library", host="127.0.0.1", port=8765)
HOURS = {"main": "08:00-22:00 on weekdays", "science": "09:00-18:00 on weekdays"}
CATALOGUE = ["Introduction to Algorithms", "Circuits Basics", "Calculus Made Easy", "Python for Engineers"]

@server.tool()
def library_hours(branch: str) -> str:
    """Opening hours of a library branch: 'main' or 'science'."""
    return HOURS.get(branch, "unknown branch")

@server.tool()
def search_library(query: str) -> str:
    """Search the library catalogue by keyword. Returns matching titles."""
    words = set(query.lower().split())
    hits = [title for title in CATALOGUE if words & set(title.lower().split())] or CATALOGUE[:2]
    return "; ".join(hits)

if __name__ == "__main__":
    server.run(transport="streamable-http")     # serves http://127.0.0.1:8765/mcp

### Step 2 — Load the server's tools and give them to the agent

The server is started once as a background process. `MultiServerMCPClient` connects to its
URL, asks it for its tools, and returns them as tool objects with the same `name`, `description`
and schema as our local tools. They mix freely with local tools in the same `ToolNode`.

In [ ]:
import subprocess, socket                                          # Python standard library
from langchain_mcp_adapters.client import MultiServerMCPClient   # langchain-mcp-adapters: MCP tools as LangChain tool objects

def start_library_server():                                       # ours: run the server as a background process, wait until it listens
    process = subprocess.Popen([sys.executable, "campus_library_server.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            socket.create_connection(("127.0.0.1", 8765), timeout=0.5).close()
            return process
        except OSError:
            time.sleep(0.5)
    raise RuntimeError("the library server did not start")

library_process = start_library_server()
print("library server running as process", library_process.pid)

mcp_client = MultiServerMCPClient({
    "library": {"transport": "streamable_http", "url": "http://127.0.0.1:8765/mcp"},   # connect to the running service
})
mcp_tools = await mcp_client.get_tools()                          # MCP: list_tools over the protocol -> tool objects
print("tools from the server:", [(t.name, t.description[:45]) for t in mcp_tools])
direct = await mcp_tools[0].ainvoke({"branch": "science"})       # MCP tools are async: ainvoke, not invoke
print("direct call         :", direct if isinstance(direct, str) else " ".join(b.get("text", "") for b in direct if isinstance(b, dict)))   # results arrive as content blocks

def agent_with_mcp(state: ChatState):                             # ours: local + remote tools in one list
    reply = model.bind_tools(KNOWLEDGE_TOOLS + mcp_tools).invoke([SystemMessage(CAMPUS_PERSONA + " Use the library tools for library questions.")] + state["messages"])   # LangChain
    return {"messages": [reply]}

g = StateGraph(ChatState)
g.add_node("agent", agent_with_mcp)
g.add_node("tools", ToolNode(KNOWLEDGE_TOOLS + mcp_tools))       # LangGraph: MCP tools run through the same node
g.add_edge(START, "agent"); g.add_conditional_edges("agent", tools_condition); g.add_edge("tools", "agent")
campusai_v11 = g.compile()

out = await campusai_v11.ainvoke({"messages": [HumanMessage("When is the science library open, and can you find a book about calculus in the library catalogue?")]})   # LangGraph: async run
show_messages(out["messages"])

### Step 3 — A public MCP server on the internet

The same client connects to servers run by other organisations. DeepWiki (by Cognition) exposes
an MCP server that answers questions about any public GitHub repository, with no key. The agent
below can ask it about LangGraph's own source. If the server is unreachable the cell says so and
moves on: a dependency on someone else's service is exactly the kind of failure G9 prepared for.

In [ ]:
import asyncio                                                    # Python standard library

async def load_public_tools():                                    # ours: connect with a timeout; an outside service may be down
    client = MultiServerMCPClient({"deepwiki": {"transport": "streamable_http", "url": "https://mcp.deepwiki.com/mcp"}})
    return await asyncio.wait_for(client.get_tools(), timeout=30)

try:
    public_tools = await load_public_tools()
    print("public server tools:", [t.name for t in public_tools])
except Exception as exc:
    public_tools = []
    print("DeepWiki not reachable right now:", type(exc).__name__, "- skipping the public-server demo")

if public_tools:
    def agent_with_public(state: ChatState):                      # ours
        reply = model.bind_tools(KNOWLEDGE_TOOLS + public_tools).invoke([SystemMessage(CAMPUS_PERSONA + " For questions about a GitHub repository, use ask_question with the repository name.")] + state["messages"])   # LangChain
        return {"messages": [reply]}
    g = StateGraph(ChatState)
    g.add_node("agent", agent_with_public)
    g.add_node("tools", ToolNode(KNOWLEDGE_TOOLS + public_tools, handle_tool_errors=True))   # LangGraph: a remote failure becomes an error message, not a crash
    g.add_edge(START, "agent"); g.add_conditional_edges("agent", tools_condition); g.add_edge("tools", "agent")
    try:
        out = await asyncio.wait_for(g.compile().ainvoke({"messages": [HumanMessage("In the langchain-ai/langgraph repository, what does a checkpointer do? Answer in two sentences.")]}), timeout=120)
        show_messages(out["messages"])
    except Exception as exc:
        print("the public server did not answer in time:", type(exc).__name__)

### Recap

- **Problem seen:** every tool had to live inside this notebook.
- **Layer added:** an MCP server process, `MultiServerMCPClient`, an async graph run with `ainvoke`, and a public MCP server with a timeout and a fallback.
- **Evidence:** the model called tools it had never seen in this notebook's code: two ran in a local process and one on a server across the internet, all through the same ToolNode.